# Step-Back Prompting for RAG
## Ask the Bigger Question First — Step-Back Prompting
⏱ ~10 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/57-step-back-prompting/step_back_workbook.ipynb)

Specific factual questions are hard to retrieve against. "What year did Einstein win the Nobel Prize?" has very different vocabulary to the document "Einstein received the Nobel Prize in Physics in 1921". **Step-back prompting** fixes this by using an LLM to transform the specific question into a broader background question before retrieval, then answering the *original* question using that broader context. By the end of this workshop you will understand *why* query abstraction improves retrieval, *how* to implement the step-back transform, and *how* to wire the full pattern into a LangGraph pipeline.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why specific queries fail, what step-back does |
| 2 | **Step-Back Transform** — LLM rewrites query to a broader question |
| 3 | **Retrieval Comparison** — Original query vs abstract query results |
| 4 | **Answer Quality** — Does step-back help answer accuracy? |
| 5 | **Full Pipeline** — step_back → retrieve → answer with LangGraph |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with `.venv` and `requirements.txt` installed, **or** Google Colab (install cell below)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Zheng, H. S. et al. (2023). *Take a Step Back: Evoking Reasoning via Abstraction in Large Language Models.* https://arxiv.org/abs/2310.06117  
> Original Google DeepMind paper proposing step-back prompting as a general reasoning strategy.

In [ ]:
# Install dependencies — runs only on Google Colab.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain",
        "langchain-openai",
        "langchain-chroma",
        "chromadb",
        "langgraph",
        "python-dotenv",
    ])
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM or embedding cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Why Specific Queries Fail

---

### The Vocabulary Mismatch Problem

Consider this specific question and the document that answers it:

```
Query:    "What year did Einstein win the Nobel Prize?"
Document: "Einstein received the Nobel Prize in Physics in 1921 for his discovery
            of the photoelectric effect."
```

Cosine similarity works on embedding space — and "What year" is conceptually distant from "received in 1921". The query is grammatically interrogative; the document is declarative. They share the keywords `Einstein` and `Nobel Prize`, but the query doesn't contain the key word `1921` or `received`.

Now consider this:

```
Abstract: "What scientific recognitions and awards did Einstein receive during his career?"
Document: "Einstein received the Nobel Prize in Physics in 1921 for his discovery
            of the photoelectric effect."
```

The abstract query is *about the same topic space* as the document — awards, recognition, career. The embedding similarity is much higher. We can then use the retrieved document to answer the original specific question.

### The Step-Back Pattern

```
ORIGINAL QUERY
  │
  │  "What year did Einstein win the Nobel Prize?"
  ▼
STEP-BACK LLM CALL
  │
  │  "What scientific recognitions and awards did Einstein receive?"  ← ABSTRACT QUERY
  ▼
RETRIEVE with ABSTRACT QUERY
  │
  │  Returns background docs about Einstein's career and achievements
  ▼
GENERATE — answer ORIGINAL QUERY from retrieved background context
  │
  │  "Einstein won the Nobel Prize in Physics in 1921."
  ▼
FINAL ANSWER
```

**Key insight:** The LLM *generates an answer-space abstraction*, not a paraphrase. It bridges the vocabulary gap between the question and the document.

In [ ]:
# Corpus and queries used throughout the workshop

SAMPLE_DOCS = [
    "Albert Einstein was born on March 14, 1879 in Ulm, Germany.",
    "Einstein developed the special theory of relativity in 1905 while working at the Swiss patent office.",
    "The general theory of relativity, published in 1915, describes gravity as spacetime curvature.",
    "Einstein received the Nobel Prize in Physics in 1921 for his discovery of the photoelectric effect.",
    "E=mc2 expresses the equivalence of energy and mass, derived from special relativity.",
    "The photoelectric effect shows that light behaves as quantized packets called photons.",
    "Quantum mechanics describes the behavior of particles at the subatomic scale.",
    "Special relativity shows that the speed of light is constant in all inertial frames.",
    "Einstein's academic career included positions at Zurich, Prague, and Princeton's IAS.",
    "The Manhattan Project, which Einstein indirectly influenced, led to atomic weapons in 1945.",
]

QUERIES = [
    "What year did Einstein win the Nobel Prize?",
    "Where did Einstein develop special relativity?",
    "What does E=mc2 mean and where does it come from?",
]

STEP_BACK_PROMPT = (
    "You are an expert at widening specific questions into broader background questions. "
    "Given a specific factual question, produce a single broader question that would retrieve "
    "the background knowledge needed to answer the original. "
    "Output only the broader question, nothing else."
)

print(f"Corpus: {len(SAMPLE_DOCS)} documents")
for i, d in enumerate(SAMPLE_DOCS):
    print(f"  [{i:02d}] {d}")

print(f"\nQueries ({len(QUERIES)}):")
for q in QUERIES:
    print(f"  • {q}")

---

## Part 2 — The Step-Back Transform

---

First build the vector store, then send each query through the step-back LLM call to produce the abstract version.

In [ ]:
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_texts(
    SAMPLE_DOCS, embeddings, collection_name="stepback-demo"
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"Index built: {len(SAMPLE_DOCS)} docs")

In [ ]:
# ─── Step-back transform — original query → abstract query ───────────────────

def step_back(query: str) -> str:
    msgs = [
        SystemMessage(content=STEP_BACK_PROMPT),
        HumanMessage(content=query),
    ]
    return llm.invoke(msgs).content.strip()


print(f"{'Original Query':<48}  Abstract Query")
print("-" * 100)

abstract_queries: dict[str, str] = {}
for q in QUERIES:
    a = step_back(q)
    abstract_queries[q] = a
    print(f"{q:<48}  {a}")

---

## Part 3 — Retrieval Comparison

---

Run retrieval twice for each query:
1. **Direct** — use the original specific question
2. **Step-back** — use the abstract question

Compare which documents each approach retrieves.

In [ ]:
K = 3  # retrieve top-3 documents

for orig_q, abs_q in abstract_queries.items():
    direct_docs = [d.page_content for d in vectorstore.similarity_search(orig_q, k=K)]
    abstract_docs = [d.page_content for d in vectorstore.similarity_search(abs_q, k=K)]

    print(f"QUERY: {orig_q}")
    print(f"ABSTRACT: {abs_q}")

    all_docs = list(dict.fromkeys(direct_docs + abstract_docs))  # preserve order, dedup
    print(f"  {'Document':<70} {'Direct':>7} {'Abstract':>8}")
    print(f"  {'─'*70} {'──────':>7} {'────────':>8}")
    for doc in all_docs:
        in_direct = "✓" if doc in direct_docs else ""
        in_abstract = "✓" if doc in abstract_docs else ""
        print(f"  {doc[:70]:<70} {in_direct:>7} {in_abstract:>8}")
    print()

---

## Part 4 — Answer Quality Comparison

---

For each query, generate two answers:
- **Direct RAG** — retrieve with original query, answer from those docs
- **Step-back RAG** — retrieve with abstract query, answer the *original* question from abstract docs

Observe whether step-back produces a more accurate or complete answer.

In [ ]:
def rag_answer(query: str, context_docs: list[str]) -> str:
    context = "\n".join(context_docs)
    prompt = (
        f"Answer the following question using ONLY the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}"
    )
    return llm.invoke([HumanMessage(content=prompt)]).content


for orig_q, abs_q in abstract_queries.items():
    direct_docs = [d.page_content for d in vectorstore.similarity_search(orig_q, k=K)]
    abstract_docs = [d.page_content for d in vectorstore.similarity_search(abs_q, k=K)]

    ans_direct = rag_answer(orig_q, direct_docs)
    ans_stepback = rag_answer(orig_q, abstract_docs)  # answer ORIGINAL from abstract context

    print(f"QUERY: {orig_q}")
    print(f"  Direct RAG   : {ans_direct}")
    print(f"  Step-back RAG: {ans_stepback}")
    print()

---

## Part 5 — Full Pipeline with LangGraph

---

Three nodes:

```
START
  │
  ▼
step_back  ─── LLM generates abstract_query from original query
  │
  ▼
retrieve   ─── Chroma: top-3 by similarity to abstract_query → documents
  │
  ▼
answer     ─── GPT-4o-mini answers ORIGINAL question from abstract context
  │
  ▼
 END
```

The `answer` node always uses the original `query` (not the `abstract_query`), ensuring we answer what was actually asked.

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class StepBackState(TypedDict):
    query:          str
    abstract_query: str
    documents:      list
    answer:         str


def step_back_node(state: StepBackState) -> dict:
    msgs = [
        SystemMessage(content=STEP_BACK_PROMPT),
        HumanMessage(content=state["query"]),
    ]
    return {"abstract_query": llm.invoke(msgs).content.strip()}


def retrieve_node(state: StepBackState) -> dict:
    docs = vectorstore.similarity_search(state["abstract_query"], k=K)
    return {"documents": [d.page_content for d in docs]}


def answer_node(state: StepBackState) -> dict:
    context = "\n".join(state["documents"])
    prompt = (
        f"Background context (retrieved using an abstracted query):\n{context}\n\n"
        f"Original question: {state['query']}\n"
        "Answer the original question using only the background context."
    )
    return {"answer": llm.invoke([HumanMessage(content=prompt)]).content}


graph = StateGraph(StepBackState)
graph.add_node("step_back", step_back_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("answer", answer_node)
graph.add_edge(START, "step_back")
graph.add_edge("step_back", "retrieve")
graph.add_edge("retrieve", "answer")
graph.add_edge("answer", END)
app = graph.compile()

print("Pipeline compiled: step_back → retrieve → answer")

In [ ]:
# ─── Run all three queries through the full pipeline ─────────────────────────

for query in QUERIES:
    print(f"{'=' * 65}")
    result: StepBackState = app.invoke(
        {"query": query, "abstract_query": "", "documents": [], "answer": ""}
    )
    print(f"Original     : {result['query']}")
    print(f"Abstract     : {result['abstract_query']}")
    print(f"Context docs : {len(result['documents'])} retrieved")
    print(f"Answer       : {result['answer']}")
    print()

---

### Exercise 1 — Customize the Step-Back Prompt

**Goal:** The STEP_BACK_PROMPT tells the LLM to produce a *broader question*. Try a different instruction:

```python
ALT_PROMPT = (
    "You are an expert at identifying the core topic of a question. "
    "Given a specific factual question, produce a single noun phrase "
    "(not a question) that captures the topic — e.g. 'Einstein Nobel Prize'. "
    "Output only the noun phrase, nothing else."
)
```

Run all three queries with `ALT_PROMPT` and compare:
1. What does the LLM generate?
2. Do retrieval results change?
3. Is the noun-phrase or the broader-question approach more effective?

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────

ALT_PROMPT = (
    "You are an expert at identifying the core topic of a question. "
    "Given a specific factual question, produce a single noun phrase "
    "(not a question) that captures the topic — e.g. 'Einstein Nobel Prize'. "
    "Output only the noun phrase, nothing else."
)

# TODO: implement step_back_alt(query, prompt) and run all 3 queries
# TODO: print original | default abstract | alt abstract
# TODO: for each, print retrieved docs and compare overlap
pass

### Exercise 2 — Chain Both Approaches

**Goal:** Instead of choosing *one* abstract query, generate *two* (default step-back + noun phrase), retrieve from each, and merge the document sets before answering.

```
query
  ├── step_back_question → retrieve → docs_q
  └── noun_phrase        → retrieve → docs_n
          └── union(docs_q, docs_n)  ← merged context
                   │
                 answer original query
```

For each query, print: how many docs came from each path, how many were unique, and what the final answer was.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

# TODO: implement dual-path retrieval
# TODO: merge docs from both paths (deduplicate)
# TODO: answer each original query from merged context
pass

---

## What's Next?

You now understand step-back prompting: transform the query to bridge the vocabulary gap, then answer the original.

### Combine with other query-rewriting techniques
- **Example 55 — HyDE** ([`../55-hyde-rag/hyde_workbook.ipynb`](../55-hyde-rag/hyde_workbook.ipynb)): instead of a broader *question*, generate a hypothetical *answer* — both are query-space transformations but with different directions
- **Example 56 — Contextual Compression** ([`../56-contextual-compression/contextual_compression_workbook.ipynb`](../56-contextual-compression/contextual_compression_workbook.ipynb)): after step-back retrieval, filter the returned docs to keep only the ones that actually answer the original question
- **Example 58 — Tabular RAG** ([`../58-tabular-rag/`](../58-tabular-rag/README.md)): when documents are structured tables rather than prose, different retrieval strategies apply

### Read the original paper
Zheng et al. (2023) propose step-back prompting as a general reasoning strategy (not just for RAG) — the LLM "steps back" to think about the principles behind a question before answering it. The RAG application here is one use case; the pattern also improves chain-of-thought reasoning on physics and math.

---

*Workshop complete. Next step: Example 58 — tabular RAG for structured data retrieval.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself.

In [ ]:
# Exercise 1 — Noun-phrase vs broader-question step-back

ALT_PROMPT = (
    "You are an expert at identifying the core topic of a question. "
    "Given a specific factual question, produce a single noun phrase "
    "(not a question) that captures the topic — e.g. 'Einstein Nobel Prize'. "
    "Output only the noun phrase, nothing else."
)


def step_back_alt(query: str, prompt: str) -> str:
    msgs = [SystemMessage(content=prompt), HumanMessage(content=query)]
    return llm.invoke(msgs).content.strip()


print(f"{'Query':<46} {'Default (broad Q)':<50} {'Alt (noun phrase)':<40}")
print("-" * 138)

for q in QUERIES:
    default_a = abstract_queries[q]  # already computed
    alt_a = step_back_alt(q, ALT_PROMPT)
    print(f"{q:<46} {default_a:<50} {alt_a:<40}")
    default_docs = vectorstore.similarity_search(default_a, k=K)
    alt_docs = vectorstore.similarity_search(alt_a, k=K)
    overlap = set(d.page_content for d in default_docs) & set(d.page_content for d in alt_docs)
    print(f"  Overlap between default and alt retrieval: {len(overlap)}/{K} docs")

In [ ]:
# Exercise 2 — Dual-path retrieval (question + noun phrase)

for q in QUERIES:
    broad_q = abstract_queries[q]
    noun_phrase = step_back_alt(q, ALT_PROMPT)

    docs_broad = [d.page_content for d in vectorstore.similarity_search(broad_q, k=K)]
    docs_noun  = [d.page_content for d in vectorstore.similarity_search(noun_phrase, k=K)]

    merged = list(dict.fromkeys(docs_broad + docs_noun))  # deduplicate, preserve order
    unique_to_broad = [d for d in docs_broad if d not in docs_noun]
    unique_to_noun  = [d for d in docs_noun  if d not in docs_broad]

    answer = rag_answer(q, merged)

    print(f"QUERY: {q}")
    print(f"  Broad-Q path  : {len(docs_broad)} docs | {len(unique_to_broad)} unique")
    print(f"  Noun-phrase   : {len(docs_noun)} docs  | {len(unique_to_noun)} unique")
    print(f"  Merged total  : {len(merged)} docs")
    print(f"  Answer        : {answer}")
    print()